In [4]:
from ebooklib import epub

book = epub.read_epub('../uploads/red-rising/book_1.epub')
title = book.get_metadata('DC', 'title')
author = book.get_metadata('DC', 'creator')

print(f'Title: {title[0][0]}')
print(f'Author: {author[0][0]}')

Title: Golden Son
Author: Pierce Brown


In [37]:
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

import time
from google.genai import errors

def get_series_via_gemini(title, author, retries=3, delay=5):
    for attempt in range(retries):
        try:
            search_response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f'Is the book "{title}" by {author} part of a series? If so, what is the book name, series name and position?',
                config=types.GenerateContentConfig(
                    tools=[types.Tool(google_search=types.GoogleSearch())], 
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )
            parse_response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"Extract the series information from this text:\n\n{search_response.text}",
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    thinking_config=types.ThinkingConfig(thinking_budget=0),
                    response_schema=types.Schema(
                        type=types.Type.OBJECT,
                        properties={
                            "is_series": types.Schema(type=types.Type.BOOLEAN),
                            "series_name": types.Schema(type=types.Type.STRING, nullable=True),
                            "position": types.Schema(type=types.Type.NUMBER, nullable=True),
                            "book_name": types.Schema(type=types.Type.STRING, nullable=True)
                        },
                        required=["is_series", "series_name", "position", "book_name"]
                    )
                )
            )
            result = parse_response.parsed

            # If claimed series but no position, treat as standalone
            if result["is_series"] and result["position"] is None:
                result["is_series"] = False
                result["series_name"] = None

            return result

        except errors.ServerError as e:
            if attempt < retries - 1:
                print(f"503 on attempt {attempt + 1}, retrying in {delay}s...")
                time.sleep(delay)
            else:
                raise

print(get_series_via_gemini("The Shadow of What Was Lost", "James Islington"))

{'is_series': True, 'series_name': 'The Licanius Trilogy', 'position': 1, 'book_name': 'The Shadow of What Was Lost'}


In [36]:
import glob
books = glob.glob('../uploads/**/*.epub', recursive=True)

for book_path in books[2:]:
    book = epub.read_epub(book_path)
    title = book.get_metadata('DC', 'title')[0][0]
    author = book.get_metadata('DC', 'creator')[0][0]
    series_info = get_series_via_gemini(title, author)
    print(f'Title: {series_info.get("book_name", title)}')
    print(f'Author: {author}')
    print(f'Is Series: {series_info["is_series"]}')
    if series_info["is_series"]:
        print(f'Series Name: {series_info["series_name"]}')
        print(f'Position in Series: {series_info["position"]}')
    print('---')

Title: An Echo of Things to Come
Author: James Islington
Is Series: True
Series Name: The Licanius Trilogy
Position in Series: 2
---
Title: The Shadow of What Was Lost
Author: James Islington
Is Series: True
Series Name: The Licanius Trilogy
Position in Series: 1
---
Title: The Light of All That Falls
Author: James Islington
Is Series: True
Series Name: The Licanius Trilogy
Position in Series: 3
---
Title: Golden Son
Author: Pierce Brown
Is Series: True
Series Name: Red Rising Saga
Position in Series: 2
---
Title: Red Rising
Author: Pierce Brown
Is Series: True
Series Name: The Red Rising Saga
Position in Series: 1
---
Title: Morning Star
Author: Pierce Brown;
Is Series: True
Series Name: The Red Rising Saga
Position in Series: 3
---
Title: The Sword of Kaigen
Author: M. L. Wang
Is Series: True
Series Name: The Theonite War
Position in Series: None
---
Title: Project Hail Mary
Author: Andy Weir
Is Series: False
---
